# Feature Engineering

### 1. Introducción

**Objetivo:** Transformar las variables del dataset preprocesado en features con mayor poder predictivo, resolviendo los problemas de codificación identificados durante la exploración.

**Dataset preprocesado:** 25492 filas, 9 columnas (salida del notebook de limpieza):
- `limite_credito`, `genero`, `educacion`, `estado_civil`, `edad`
- `meses_deuda_sep`, `pago_sep`, `factura_sep`
- `default_oct` (target)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import src.data.preprocess as pp
import src.features.feature_engineering as fe

In [ ]:
df = pp.preprocesar_datos()
print(f'Shape: {df.shape}')
df.head()

---
### 2. Descomposición de `meses_deuda_sep`

La columna `meses_deuda_sep` presenta un problema de codificación: usa una misma escala entera para representar conceptos cualitativamente distintos.

| Valor | Significado |
|-------|-------------|
| -2 | Sin consumo en el mes |
| -1 | Usó crédito y pagó el total a tiempo |
| 0 | Pago mínimo (no entró en mora) |
| 1, 2, … | Meses de atraso acumulados |

Tratar estos valores como una variable numérica continua haría que el modelo interprete, por ejemplo, que `-2` y `2` son opuestos simétricos, cuando en realidad representan comportamientos completamente distintos. Por eso descomponemos la columna en features separadas:

In [ ]:
df = fe.preprocesar_meses_deuda(df)

nuevas_cols = ['tiene_deuda_sep', 'n_meses_deuda_sep', 'sin_uso_sep', 'pago_minimo_sep', 'pago_completo_sep']
df[nuevas_cols + ['default_oct']].head(10)

Las nuevas features y su interpretación:

- `tiene_deuda_sep` — indicador binario: el cliente tiene al menos 1 mes de atraso.
- `n_meses_deuda_sep` — cantidad de meses adeudados (0 si no tiene deuda real).
- `sin_uso_sep` — no realizó consumos durante el mes.
- `pago_minimo_sep` — realizó el pago mínimo sin entrar en mora.
- `pago_completo_sep` — pagó el total de la factura a tiempo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colores = ['#4F69D9', '#CC403A']
labels_default = {0: 'No Default', 1: 'Default'}

# Distribución de n_meses_deuda_sep según default
for val, label in labels_default.items():
    axes[0].hist(
        df[df['default_oct'] == val]['n_meses_deuda_sep'],
        bins=range(0, 10),
        alpha=0.6,
        color=colores[val],
        label=label,
        density=True
    )
axes[0].set_title('Distribución de meses de deuda según Default')
axes[0].set_xlabel('n_meses_deuda_sep')
axes[0].set_ylabel('Densidad')
axes[0].legend()

# Tasa de default según tiene_deuda_sep
tasa_default = df.groupby('tiene_deuda_sep')['default_oct'].mean()
axes[1].bar(
    ['Sin deuda (0)', 'Con deuda (1)'],
    tasa_default.values,
    color=colores
)
axes[1].set_title('Tasa de default según tiene_deuda_sep')
axes[1].set_ylabel('Proporción de defaults')
axes[1].set_ylim(0, 1)
for i, v in enumerate(tasa_default.values):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

Los clientes con deuda acumulada tienen una tasa de default considerablemente mayor. La separación entre las distribuciones confirma el poder predictivo de `n_meses_deuda_sep`.

---
### 3. Creación de ratios financieros

Además de las variables originales, construimos dos ratios que capturan el **comportamiento relativo** del cliente, más informativo que los montos absolutos:

- **`ratio_uso_sep`** = `factura_sep / limite_credito` — qué proporción del crédito disponible utilizó el cliente. Un ratio alto puede indicar mayor presión financiera.
- **`ratio_pago_sep`** = `pago_sep / (factura_sep + 1)` — qué fracción de la factura pagó. Un ratio bajo puede indicar dificultades para afrontar las deudas.

In [ ]:
df = fe.crear_features_avanzadas(df)

print('Estadísticas de los nuevos ratios:')
df[['ratio_uso_sep', 'ratio_pago_sep']].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col, titulo in zip(
    axes,
    ['ratio_uso_sep', 'ratio_pago_sep'],
    ['Ratio de utilización de crédito', 'Ratio de pago sobre factura']
):
    for val, label in labels_default.items():
        ax.hist(
            df[df['default_oct'] == val][col],
            bins=30,
            alpha=0.6,
            color=colores[val],
            label=label,
            density=True
        )
    ax.set_title(titulo)
    ax.set_xlabel(col)
    ax.set_ylabel('Densidad')
    ax.legend()

plt.tight_layout()
plt.show()

---
### 4. Dataset final

Luego del feature engineering, el dataset pasa de 9 columnas a 13 features más el target:

In [ ]:
print(f'Shape final: {df.shape}')
print(f'\nColumnas: {df.columns.tolist()}')
df.head()

In [ ]:
# Correlación de todas las features con el target
corr_target = df.corr()['default_oct'].drop('default_oct').sort_values()

fig, ax = plt.subplots(figsize=(7, 5))
colores_barras = ['#CC403A' if v > 0 else '#4F69D9' for v in corr_target.values]
ax.barh(corr_target.index, corr_target.values, color=colores_barras)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlación de cada feature con default_oct')
ax.set_xlabel('Correlación de Pearson')
plt.tight_layout()
plt.show()

`n_meses_deuda_sep` y `tiene_deuda_sep` son las features con mayor correlación positiva con el target, mientras que `limite_credito` mantiene la correlación negativa más pronunciada — consistente con el análisis exploratorio del notebook anterior.

### 5. Próximos pasos

Con el dataset transformado, el siguiente paso es el entrenamiento y evaluación de modelos de clasificación, priorizando el **recall** como métrica principal dado el costo asimétrico de los falsos negativos en el contexto de riesgo crediticio.